# Ejercicio 1 · Portátiles: ¿rompen los ensembles el empate?


**Módulo 5072 · SAA · UD3 · Métodos ensemble** · 
Una tienda quiere saber el precio justo de un portátil a partir de sus características. Hoy probamos Random Forest y XGBoost.

- **Qué predecimos:** el precio en rupias (`Price`). Es **regresión**.
- **Métrica:** el **MAE**, cuántas rupias se equivoca el modelo de media.

**Qué vas a hacer:** entrenar cuatro modelos, compararlos en una tabla y elegir uno.

**Cómo trabajar:**
- Las celdas que dicen **«ya está hecha»**: solo ejecútalas y lee el resultado.
- Las celdas de **«Tu turno»**: cambia cada `___` por lo que falta. Si te atascas, abre la pista.

## 1 · Preparación

Esta celda ya está hecha: ejecútala.

In [20]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

DATA = "../data/"
SEMILLA = 42

# 1. Cargar los datos y quitar las filas repetidas
df = pd.read_csv(DATA + "laptop.csv").drop_duplicates()
print(df.shape)



(1273, 15)


In [21]:
# 2. separamos la variable objetivo del resto de columnas
X = df.drop("Price", axis=1)
y = df["Price"]



In [22]:
# 3. Apartamos el 20 % para el test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,  random_state=SEMILLA)
print(X_train.shape, X_test.shape)



(1018, 14) (255, 14)


In [23]:
# 4. No hay huecos. Las columnas de texto pasan a números
texto = ["Company", "TypeName", "OpSys", "CpuCompany", "GPU"]

# Texto a números con OneHotEncoder: aprende las categorías SOLO con el train
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(X_train[texto])

def codificar(X):
    cod = pd.DataFrame(ohe.transform(X[texto]), columns=ohe.get_feature_names_out(), index=X.index)
    return pd.concat([X.drop(columns=texto), cod], axis=1)

X_train = codificar(X_train)
X_test = codificar(X_test)




In [24]:

# 5. Validación cruzada: el train en 5 trozos
cv = KFold(n_splits=5, shuffle=True, random_state=SEMILLA)



#  evaluar(): validación cruzada. Entrena el modelo 5 veces, cada vez con 4 trozos,
#    y mide el error en el trozo que queda fuera (validación). El test no se toca.
def evaluar(nombre, modelo):
    r = cross_validate(modelo, X_train, y_train, cv=cv,
                       scoring="neg_mean_absolute_error", return_train_score=True)
    # scikit-learn da el MAE en negativo; con el - delante lo ponemos en positivo
    return {"modelo": nombre,
            "MAE val": -r["test_score"].mean(),
            "± MAE val": r["test_score"].std(),
            "MAE train": -r["train_score"].mean()}

def mostrar(resultados):
    return pd.DataFrame(resultados).round(0)

In [25]:
# 7. Baseline: el modelo más tonto, que dice siempre el precio medio del train
pred_base = np.full(len(y_train), y_train.mean())
resultados = [{"modelo": "Baseline (la media)", "MAE val": mean_absolute_error(y_train, pred_base),
               "± MAE val": np.nan, "MAE train": mean_absolute_error(y_train, pred_base)}]
print("Listo:", len(X_train), "portátiles para entrenar y validar ·", len(X_test), "para el test")

Listo: 1018 portátiles para entrenar y validar · 255 para el test


**Qué ha hecho la celda**
1. Carga los portátiles y quita las filas repetidas.
2. Separa `X` (las pistas) e `y` (el precio).
3. Aparta el 20 % para el **test**, que no tocamos hasta el final.
4. Pasa las columnas de texto (marca, tipo, sistema...) a columnas de 0 y 1 con `OneHotEncoder`, aprendido solo con el train.
5. Prepara la **validación cruzada** de 5 trozos.
6. Crea `evaluar()` y `mostrar()`, como en el gimnasio.
7. Crea la tabla `resultados` con el **baseline**.

**Cómo leer la tabla**
- `MAE val`: cuántas rupias se equivoca de media con portátiles que no ha visto (los trozos de validación). Es la columna que usamos para comparar: cuanto **menos**, mejor.
- `± MAE val`: cuánto cambia de un trozo de validación a otro. Si dos modelos se diferencian menos que esto, es un **empate**.
- `MAE train`: lo mismo con los datos que el modelo ya ha visto. Si es mucho más bajo que `MAE val`, el modelo **memoriza**.

In [26]:
mostrar(resultados)

,modelo,MAE val,± MAE val,MAE train
0,Baseline (la media),28521.0,NaN,28521.0


**Fíjate:** ¿Cuántas rupias se equivoca de media el baseline? 

*Tu respuesta:*

Se equivoca de media en 28521 rupias

## 2 · Tu turno: entrena y compara

**Tarea 1 · Un árbol sin límite.** Cambia `___` por el modelo de árbol: `DecisionTreeRegressor`.

In [27]:
resultados.append(evaluar("Árbol sin límite", DecisionTreeRegressor(random_state=SEMILLA)))
mostrar(resultados)

,modelo,MAE val,± MAE val,MAE train
0,Baseline (la media),28521.0,NaN,28521.0
1,Árbol sin límite,13518.0,373.0,429.0


**Fíjate:** Compara `MAE train` y `MAE val`. ¿Memoriza?

*Tu respuesta:*

Sí, memoriza. MAE train 429 es muy bajo y MAE val 13.518 es mucho más alto. El árbol sin límite se aprende los datos de memoria y no generaliza bien, aunque ya mejora al baseline

**Tarea 2 · Random Forest de 300 árboles.** Cambia los `___` por RandomForestRegressor

<details><summary><b>Pista</b></summary>

El modelo es `RandomForestRegressor` y el número de árboles va en `n_estimators`.

</details>

In [28]:
resultados.append(evaluar("Random Forest", RandomForestRegressor(n_estimators=300, random_state=SEMILLA, n_jobs=-1)))
mostrar(resultados)

,modelo,MAE val,± MAE val,MAE train
0,Baseline (la media),28521.0,NaN,28521.0
1,Árbol sin límite,13518.0,373.0,429.0
2,Random Forest,10727.0,756.0,4130.0


**Fíjate:** ¿Mejora al árbol? ¿La diferencia es mayor que el `±`?

Si mejora al arbol sin limite. Baja de 13518 a 10727, además reduce el overfitting.

*Tu respuesta:*

**Tarea 3 · XGBoost por defecto.** Cambia `___` por `XGBRegressor`.

In [29]:
resultados.append(evaluar("XGBoost por defecto", XGBRegressor(random_state=SEMILLA)))
mostrar(resultados)

,modelo,MAE val,± MAE val,MAE train
0,Baseline (la media),28521.0,NaN,28521.0
1,Árbol sin límite,13518.0,373.0,429.0
2,Random Forest,10727.0,756.0,4130.0
3,XGBoost por defecto,10724.0,602.0,2045.0


**Fíjate:** Compara `MAE train` y `MAE val`. ¿Memoriza?

*Tu respuesta:*

Si, memoriza. Train 2045 es mucho más bajo que 10724. Y comparado con random forest es empate


**Tarea 4 · XGBoost con frenos.** Pon:
- `max_depth=3`: árboles pequeños.
- `learning_rate=0.05`: poco caso a cada árbol.
- `n_estimators=300`: como le hacemos poco caso a cada uno, hacen falta más árboles.

In [30]:
xgb_frenos =  XGBRegressor(max_depth=3, learning_rate=0.05, n_estimators=300, random_state=SEMILLA)
resultados.append(evaluar("XGBoost con frenos", xgb_frenos))
mostrar(resultados)

,modelo,MAE val,± MAE val,MAE train
0,Baseline (la media),28521.0,NaN,28521.0
1,Árbol sin límite,13518.0,373.0,429.0
2,Random Forest,10727.0,756.0,4130.0
3,XGBoost por defecto,10724.0,602.0,2045.0
4,XGBoost con frenos,10943.0,550.0,8137.0


**Fíjate:** Compáralo con el XGBoost por defecto. ¿Memoriza menos? ¿Mejora `MAE val`?

*Tu respuesta:*

Sí, memoriza mucho menos. Pasa de train 2.045 a 8.137, el hueco baja de 8.679 a 2.806 y el ratio sube a 0.74 que es el rango bueno. Pero no mejora el MAE val, sube de 10.724 a 10.943, son 219 rupias de diferencia que es menos que el ±, así que es empate o un poco peor.

## 3 · ¿Cuál elegimos?

Esta celda ya está hecha: ejecútala. Ordena la tabla: el mejor queda arriba.

In [31]:
mostrar(pd.DataFrame(resultados).sort_values("MAE val", ascending=True))

,modelo,MAE val,± MAE val,MAE train
3,XGBoost por defecto,10724.0,602.0,2045.0
2,Random Forest,10727.0,756.0,4130.0
4,XGBoost con frenos,10943.0,550.0,8137.0
1,Árbol sin límite,13518.0,373.0,429.0
0,Baseline (la media),28521.0,NaN,28521.0


Contesta:
1. ¿Qué modelo gana?
2. Resta su `MAE val` y el del segundo. ¿La diferencia es mayor que el `±`? ¿Gana o es un empate?
3. ¿Cuál eliges? Si hay empate, elige el más sencillo o el más rápido.

*Tu respuesta:*

Gana XGBoost por defecto con 10.724. 2. La diferencia con el segundo [Random Forest 10.727] es de solo 3 rupias, que es mucho menor que el ± de 602 y 756, así que es un empate. 3. Elijo XGBoost por defecto porque empata con Random Forest pero es más sencillo y rápido

## 4 · El test, una sola vez

Esta celda ya está hecha: ejecútala. Entrena el Random Forest y el XGBoost con frenos con todo el train y los prueba en el test, que hasta ahora no habíamos tocado.

In [32]:
rf = RandomForestRegressor(n_estimators=300, random_state=SEMILLA, n_jobs=-1).fit(X_train, y_train)
xgb_frenos.fit(X_train, y_train)

print("Random Forest      · MAE test:", round(mean_absolute_error(y_test, rf.predict(X_test))))
print("XGBoost con frenos · MAE test:", round(mean_absolute_error(y_test, xgb_frenos.predict(X_test))))

Random Forest      · MAE test: 9848
XGBoost con frenos · MAE test: 10659


**Fíjate:** ¿El resultado del test se parece al de validación (`MAE val`)? Si se parece, la validación nos dijo la verdad.

*Tu respuesta:*

Sí, se parece. Random Forest val 10727 y test 9848, XGBoost con frenos val 10943 y test 10659. Los dos dan parecido, un poco menos en test pero están en el mismo rango de 10k, así que la validación era buena.

## 5 · Conclusión

¿qué modelo eliges? ¿Por qué? Usa números de la tabla.

*Tu respuesta:*

Yo elijo el Random Forest, y el XGBoost por defecto está empatado.

En la tabla ordenada el mejor era XGBoost por defecto con 10724 y Random Forest con 10727, la diferencia es de 3 rupias que es mucho menos que el ± de 602 y 756 así que es empate. Los dos bajan mucho respecto al baseline que era 28521 y al árbol sin límite que era 13518 con train 429 que era overfitting total.

En el test el Random Forest da 9848 y el XGBoost con frenos 10659, el Random Forest generaliza un poco mejor. El XGBoost con frenos tenía train 8137 y val 10943 que ya no memorizaba tanto pero su val era peor.

Así que me quedo con Random Forest porque de 28521 del baseline lo baja a 9848 en test y es el que mejor queda. Si hay que elegir uno rápido me quedo con XGBoost por defecto que es 10724 con solo 100 árboles.

## 6 · 


**Extra 1 · ¿Cuántos árboles hacen falta?** Esta celda ya está hecha: ejecútala. Entrena Random Forest con 10, 50 y 100 árboles.

In [33]:
extra = []
for n in [10, 50, 100]:
    extra.append(evaluar(f"Random Forest de {n} árboles", RandomForestRegressor(n_estimators=n, random_state=SEMILLA, n_jobs=-1)))
mostrar(extra)

,modelo,MAE val,± MAE val,MAE train
0,Random Forest de 10 árboles,11097.0,587.0,4574.0
1,Random Forest de 50 árboles,10700.0,700.0,4212.0
2,Random Forest de 100 árboles,10744.0,713.0,4182.0


**Fíjate:** ¿Mejora al pasar de 10 a 50 árboles? ¿Y de 50 a 100? ¿Empeora alguna vez al poner más árboles?

XGBOOST.** El mismo XGBoost con frenos, pero con `learning_rate=0.5`  `learning_rate=0.05`  en lugar de 0,05.

In [34]:
xgb_rapido = XGBRegressor(max_depth=3, learning_rate=0.05, n_estimators=300, random_state=SEMILLA)
extra.append(evaluar("XGBoost con learning_rate 0,05", xgb_rapido))
mostrar(extra)

,modelo,MAE val,± MAE val,MAE train
0,Random Forest de 10 árboles,11097.0,587.0,4574.0
1,Random Forest de 50 árboles,10700.0,700.0,4212.0
2,Random Forest de 100 árboles,10744.0,713.0,4182.0
3,"XGBoost con learning_rate 0,05",10943.0,550.0,8137.0


In [35]:
xgb_rapido = XGBRegressor(max_depth=3, learning_rate=0.5, n_estimators=300, random_state=SEMILLA)
extra.append(evaluar("XGBoost con learning_rate 0,5", xgb_rapido))
mostrar(extra)

,modelo,MAE val,± MAE val,MAE train
0,Random Forest de 10 árboles,11097.0,587.0,4574.0
1,Random Forest de 50 árboles,10700.0,700.0,4212.0
2,Random Forest de 100 árboles,10744.0,713.0,4182.0
3,"XGBoost con learning_rate 0,05",10943.0,550.0,8137.0
4,"XGBoost con learning_rate 0,5",10510.0,716.0,3072.0


**Fíjate:** Compáralo con el XGBoost con frenos (0,05). ¿Memoriza más? ¿Mejora o empeora `MAE val`?

*Tu respuesta:*

Con lr 0,5 memoriza más, el train baja de 8137 a 3072 y el hueco sube. Pero el MAE val mejora de 10943 a 10510, baja 433 que está justo por debajo del ± así que mejora un poco pero sigue siendo casi empate